# Preparacao e modelagem dimensional dos acidentes da PRF

## 1. Introducao

Este notebook prepara a base de acidentes da PRF para analises futuras. O foco e:

- validar schemas e qualidade dos datasets carregados;
- processar grandes volumes por leitura em partes (`chunks`);
- organizar os dados em uma tabela fato de acidentes e tabelas auxiliares/dimensoes;
- conferir especificamente as causas dos acidentes com apoio do dicionario de dados.

A organizacao segue a ideia da imagem `foto.jpeg`: a tabela fato centraliza chaves e metricas, enquanto tempo e localidade funcionam como dimensoes N:1 e veiculos, envolvidos, circunstancias e causas preservam os detalhes 1:N ligados pelo campo original `id`.

> As colunas originais nao sao renomeadas. As unicas colunas novas sao chaves tecnicas prefixadas por `sk_` e metricas derivadas claramente identificadas.

## 2. Importacoes e configuracoes

Sao reutilizados o schema e as regras existentes em `src/etl/schema.py`. Para grandes volumes, a leitura usa Pandas em partes e PyArrow/Parquet fica disponivel para salvamento opcional.

In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import csv
import hashlib
import sys
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 30)

def localizar_raiz() -> Path:
    atual = Path.cwd().resolve()
    for candidato in [atual, atual.parent]:
        if (candidato / "data" / "raw").exists() and (candidato / "src").exists():
            return candidato
    raise FileNotFoundError("Execute o notebook a partir da raiz do projeto ou da pasta notebooks.")

ROOT = localizar_raiz()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.etl.schema import (
    DATE_COLUMNS,
    NUMERIC_NON_NEGATIVE_COLUMNS,
    SCHEMA_PANDAS,
    TIME_COLUMNS,
    validate_required_columns,
)

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
CSV_PATHS = sorted(RAW_DIR.glob("acidentes*_todas_causas_tipos.csv"))
CHUNKSIZE = 100_000

print(f"Raiz: {ROOT}")
print(f"Arquivos CSV encontrados: {len(CSV_PATHS)}")

Raiz: /home/chrys/Downloads/pad_acidentes/pad_acidentes_prf-main
Arquivos CSV encontrados: 9


## 3. Carregamento dos dados e do dicionario

Nao ha um arquivo local de dicionario nesta copia do projeto. Por isso, o dataframe abaixo consolida um **dicionario operacional** com os nomes reais presentes nos CSVs, os tipos semanticos esperados e seus significados. As definicoes seguem a nomenclatura oficial da PRF e sao usadas nas validacoes.

Fonte de referencia: [Dicionario de dados de acidentes da PRF](https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dicionario-acidentes).

In [2]:
DICIONARIO_DADOS = [
    ("id", "inteiro", "Identificador do acidente."),
    ("pesid", "inteiro", "Identificador da pessoa envolvida."),
    ("data_inversa", "data", "Data da ocorrencia no formato ano-mes-dia."),
    ("dia_semana", "categoria", "Dia da semana da ocorrencia."),
    ("horario", "hora", "Horario da ocorrencia."),
    ("uf", "categoria", "Unidade da Federacao."),
    ("br", "inteiro", "Numero da rodovia federal."),
    ("km", "decimal", "Quilometro da rodovia onde ocorreu o acidente."),
    ("municipio", "categoria", "Municipio da ocorrencia."),
    ("causa_principal", "categoria", "Indica se a causa registrada e a causa principal."),
    ("causa_acidente", "categoria", "Causa presumivel do acidente."),
    ("ordem_tipo_acidente", "inteiro", "Ordem do tipo de acidente no registro."),
    ("tipo_acidente", "categoria", "Tipo ou configuracao do acidente."),
    ("classificacao_acidente", "categoria", "Classificacao do acidente segundo a gravidade."),
    ("fase_dia", "categoria", "Fase do dia no momento do acidente."),
    ("sentido_via", "categoria", "Sentido da via."),
    ("condicao_metereologica", "categoria", "Condicao meteorologica informada."),
    ("tipo_pista", "categoria", "Tipo da pista."),
    ("tracado_via", "categoria", "Caracteristica do tracado da via."),
    ("uso_solo", "categoria", "Indica uso do solo/trecho urbano."),
    ("id_veiculo", "inteiro", "Identificador do veiculo envolvido."),
    ("tipo_veiculo", "categoria", "Tipo do veiculo."),
    ("marca", "texto", "Marca/modelo informado do veiculo."),
    ("ano_fabricacao_veiculo", "inteiro", "Ano de fabricacao do veiculo."),
    ("tipo_envolvido", "categoria", "Papel da pessoa envolvida."),
    ("estado_fisico", "categoria", "Estado fisico da pessoa envolvida."),
    ("idade", "inteiro", "Idade da pessoa envolvida."),
    ("sexo", "categoria", "Sexo informado da pessoa envolvida."),
    ("ilesos", "inteiro", "Quantidade de ilesos no acidente."),
    ("feridos_leves", "inteiro", "Quantidade de feridos leves no acidente."),
    ("feridos_graves", "inteiro", "Quantidade de feridos graves no acidente."),
    ("mortos", "inteiro", "Quantidade de mortos no acidente."),
    ("latitude", "decimal", "Latitude da ocorrencia."),
    ("longitude", "decimal", "Longitude da ocorrencia."),
    ("regional", "categoria", "Superintendencia regional responsavel."),
    ("delegacia", "categoria", "Delegacia responsavel."),
    ("uop", "categoria", "Unidade operacional responsavel."),
]

df_dicionario = pd.DataFrame(DICIONARIO_DADOS, columns=["coluna", "tipo_esperado", "definicao"])
COLUNAS_DICIONARIO = df_dicionario["coluna"].tolist()
df_dicionario.head(10)

,coluna,tipo_esperado,definicao
0,id,inteiro,Identificador do acidente.
1,pesid,inteiro,Identificador da pessoa envolvida.
2,data_inversa,data,Data da ocorrencia no formato ano-mes-dia.
3,dia_semana,categoria,Dia da semana da ocorrencia.
4,horario,hora,Horario da ocorrencia.
5,uf,categoria,Unidade da Federacao.
6,br,inteiro,Numero da rodovia federal.
7,km,decimal,Quilometro da rodovia onde ocorreu o acidente.
8,municipio,categoria,Municipio da ocorrencia.
9,causa_principal,categoria,Indica se a causa registrada e a causa principal.


In [3]:
def cabecalho_csv(caminho: Path) -> list[str]:
    return pd.read_csv(caminho, sep=";", encoding="latin1", nrows=0).columns.tolist()

df_arquivos = pd.DataFrame(
    {
        "arquivo": [p.name for p in CSV_PATHS],
        "tamanho_mb": [round(p.stat().st_size / 1024**2, 2) for p in CSV_PATHS],
        "qtd_colunas": [len(cabecalho_csv(p)) for p in CSV_PATHS],
    }
)
df_arquivos

,arquivo,tamanho_mb,qtd_colunas
0,acidentes2017_todas_causas_tipos.csv,119.12,37
1,acidentes2018_todas_causas_tipos.csv,111.84,37
2,acidentes2019_todas_causas_tipos.csv,119.60,37
3,acidentes2020_todas_causas_tipos.csv,142.36,37
4,acidentes2021_todas_causas_tipos.csv,162.83,37
5,acidentes2022_todas_causas_tipos.csv,182.90,37
6,acidentes2023_todas_causas_tipos.csv,206.05,37
7,acidentes2024_todas_causas_tipos.csv,222.29,37
8,acidentes2025_todas_causas_tipos.csv,128.86,37


## 4. Definicao dos schemas

O schema original de `src/etl/schema.py` e preservado. Abaixo ele e ampliado para todas as colunas do dicionario. Na leitura bruta, os campos entram inicialmente como texto para tolerar variacoes entre anos; depois, datas e numeros sao convertidos com `errors="coerce"`, permitindo contar valores incompatíveis sem interromper o processamento.

In [4]:
TIPOS_INTEIROS = df_dicionario.query("tipo_esperado == 'inteiro'")["coluna"].tolist()
TIPOS_DECIMAIS = df_dicionario.query("tipo_esperado == 'decimal'")["coluna"].tolist()
TIPOS_DATAS = df_dicionario.query("tipo_esperado == 'data'")["coluna"].tolist()
TIPOS_HORAS = df_dicionario.query("tipo_esperado == 'hora'")["coluna"].tolist()

SCHEMA_COMPLETO = dict(zip(df_dicionario["coluna"], df_dicionario["tipo_esperado"]))
COLUNAS_CRITICAS = ["id", "data_inversa", "uf", "municipio", "causa_acidente"]

print("Schema reaproveitado de src/etl/schema.py:")
print(SCHEMA_PANDAS)
print(f"Schema ampliado: {len(SCHEMA_COMPLETO)} colunas")

Schema reaproveitado de src/etl/schema.py:
{'id': 'Int64', 'dia_semana': 'category', 'uf': 'category', 'br': 'string', 'municipio': 'category', 'causa_acidente': 'category', 'tipo_acidente': 'category', 'fase_dia': 'category', 'condicao_metereologica': 'category', 'tipo_pista': 'category', 'tracado_via': 'category', 'ilesos': 'Int16', 'feridos_leves': 'Int16', 'feridos_graves': 'Int16', 'mortos': 'Int8'}
Schema ampliado: 37 colunas


## 5. Validacao dos schemas

As verificacoes cobrem colunas obrigatorias, ausentes, extras, compatibilidade com o dicionario, tipos convertiveis e nulos em campos criticos. A validacao completa percorre todos os arquivos em partes, sem concatenar a base bruta.

In [5]:
def iterar_chunks(colunas: list[str] | None = None):
    for caminho in CSV_PATHS:
        presentes = cabecalho_csv(caminho)
        usar = [c for c in (colunas or presentes) if c in presentes]
        for chunk in pd.read_csv(
            caminho,
            sep=";",
            encoding="latin1",
            dtype="string",
            chunksize=CHUNKSIZE,
            on_bad_lines="skip",
        ):
            yield caminho.name, chunk[usar]


def converter_tipos(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for coluna in TIPOS_INTEIROS:
        if coluna in df:
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce").astype("Int64")
    for coluna in TIPOS_DECIMAIS:
        if coluna in df:
            normalizada = df[coluna].str.replace(",", ".", regex=False)
            df[coluna] = pd.to_numeric(normalizada, errors="coerce")
    if "data_inversa" in df:
        df["data_inversa"] = pd.to_datetime(df["data_inversa"], errors="coerce")
    return df


def validar_em_partes() -> tuple[pd.DataFrame, pd.DataFrame]:
    nulos = Counter()
    invalidos = Counter()
    total_linhas = 0

    for _, chunk in iterar_chunks(COLUNAS_DICIONARIO):
        total_linhas += len(chunk)
        nulos.update(chunk.isna().sum().to_dict())

        for coluna in TIPOS_INTEIROS + TIPOS_DECIMAIS:
            serie = chunk[coluna]
            normalizada = serie.str.replace(",", ".", regex=False)
            invalidos[coluna] += int((serie.notna() & pd.to_numeric(normalizada, errors="coerce").isna()).sum())

        datas = pd.to_datetime(chunk["data_inversa"], errors="coerce")
        invalidos["data_inversa"] += int((chunk["data_inversa"].notna() & datas.isna()).sum())

        horas = pd.to_datetime(chunk["horario"], format="%H:%M:%S", errors="coerce")
        invalidos["horario"] += int((chunk["horario"].notna() & horas.isna()).sum())

    rel_nulos = pd.DataFrame(
        {
            "coluna": COLUNAS_DICIONARIO,
            "qtd_nulos": [nulos[c] for c in COLUNAS_DICIONARIO],
        }
    )
    rel_nulos["percentual_nulos"] = (rel_nulos["qtd_nulos"] / total_linhas * 100).round(4)

    rel_tipos = df_dicionario[["coluna", "tipo_esperado"]].copy()
    rel_tipos["valores_nao_convertiveis"] = rel_tipos["coluna"].map(invalidos).fillna(0).astype("int64")
    rel_tipos["status"] = np.where(rel_tipos["valores_nao_convertiveis"].eq(0), "ok", "verificar")
    return rel_nulos, rel_tipos


df_validacao_colunas = validate_required_columns(CSV_PATHS, required_columns=COLUNAS_DICIONARIO)
headers_unidos = set().union(*(set(cabecalho_csv(p)) for p in CSV_PATHS))
df_colunas_extras = pd.DataFrame({"coluna_extra": sorted(headers_unidos - set(COLUNAS_DICIONARIO))})
df_compatibilidade_dicionario = df_dicionario.assign(
    presente_no_dataset=df_dicionario["coluna"].isin(headers_unidos)
)

df_validacao_nulos, df_validacao_tipos = validar_em_partes()
df_nulos_criticos = df_validacao_nulos.query("coluna in @COLUNAS_CRITICAS")

print("Colunas ausentes por arquivo:")
display(df_validacao_colunas)
print("Colunas extras:")
display(df_colunas_extras if not df_colunas_extras.empty else pd.DataFrame({"mensagem": ["Nenhuma coluna extra."]}))
print("Nulos em campos criticos:")
display(df_nulos_criticos)
print("Conversoes que exigem verificacao:")
display(df_validacao_tipos.query("status != 'ok'"))

Colunas ausentes por arquivo:


,arquivo,colunas_esperadas,colunas_ausentes,lista_colunas_ausentes
0,acidentes2017_todas_causas_tipos.csv,37,0,Nenhuma
1,acidentes2018_todas_causas_tipos.csv,37,0,Nenhuma
2,acidentes2019_todas_causas_tipos.csv,37,0,Nenhuma
3,acidentes2020_todas_causas_tipos.csv,37,0,Nenhuma
4,acidentes2021_todas_causas_tipos.csv,37,0,Nenhuma
5,acidentes2022_todas_causas_tipos.csv,37,0,Nenhuma
6,acidentes2023_todas_causas_tipos.csv,37,0,Nenhuma
7,acidentes2024_todas_causas_tipos.csv,37,0,Nenhuma
8,acidentes2025_todas_causas_tipos.csv,37,0,Nenhuma


Colunas extras:


,mensagem
0,Nenhuma coluna extra.


Nulos em campos criticos:


,coluna,qtd_nulos,percentual_nulos
0,id,0,0.0000
2,data_inversa,38,0.0010
5,uf,55,0.0014
8,municipio,28298,0.7316
10,causa_acidente,6019,0.1556


Conversoes que exigem verificacao:


,coluna,tipo_esperado,valores_nao_convertiveis,status
0,id,inteiro,28328,verificar
1,pesid,inteiro,6047,verificar
2,data_inversa,data,28360,verificar
4,horario,hora,28367,verificar
6,br,inteiro,22399,verificar
7,km,decimal,22417,verificar
11,ordem_tipo_acidente,inteiro,68,verificar
20,id_veiculo,inteiro,26,verificar
23,ano_fabricacao_veiculo,inteiro,23,verificar
26,idade,inteiro,18,verificar


## 6. Processamento em grandes volumes

Cada tabela e construida por uma nova passagem em chunks. Em cada parte, duplicatas locais sao removidas; ao final, ocorre uma deduplicacao global. Essa estrategia troca um pouco de tempo de leitura por menor pico de memoria e evita um dataframe bruto unico com milhoes de linhas e todas as colunas.

In [6]:
def compactar_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    for coluna in df.select_dtypes(include=["string", "object"]).columns:
        if len(df) and df[coluna].nunique(dropna=False) / len(df) < 0.50:
            df[coluna] = df[coluna].astype("category")
    return df


def carregar_unicos(colunas: list[str], subset: list[str], filtrar: str | None = None) -> pd.DataFrame:
    partes = []
    for _, chunk in iterar_chunks(colunas):
        chunk = converter_tipos(chunk)
        if filtrar:
            chunk = chunk[chunk[filtrar].notna()]
        partes.append(chunk.drop_duplicates(subset=subset))

    resultado = pd.concat(partes, ignore_index=True).drop_duplicates(subset=subset).reset_index(drop=True)
    return compactar_dataframe(resultado)


COLUNAS_ACIDENTE = [
    "id", "data_inversa", "dia_semana", "horario", "uf", "br", "km", "municipio",
    "classificacao_acidente", "fase_dia", "latitude", "longitude", "regional", "delegacia", "uop",
    "ilesos", "feridos_leves", "feridos_graves", "mortos",
]

def carregar_acidentes_agregados() -> pd.DataFrame:
    partes = []
    metricas = ["ilesos", "feridos_leves", "feridos_graves", "mortos"]
    atributos = [c for c in COLUNAS_ACIDENTE if c not in ["id", *metricas]]

    for _, chunk in iterar_chunks(COLUNAS_ACIDENTE):
        chunk = converter_tipos(chunk)
        chunk = chunk[chunk["id"].notna()]
        grupo = chunk.groupby("id", observed=True)
        parte = grupo[atributos].first().join(grupo[metricas].max()).reset_index()
        partes.append(parte)

    combinado = pd.concat(partes, ignore_index=True)
    grupo_final = combinado.groupby("id", observed=True)
    resultado = grupo_final[atributos].first().join(grupo_final[metricas].max()).reset_index()
    resultado["data_hora"] = pd.to_datetime(
        resultado["data_inversa"].dt.strftime("%Y-%m-%d") + " " + resultado["horario"].astype("string"),
        errors="coerce",
    )
    return compactar_dataframe(resultado)


df_acidentes_base = carregar_acidentes_agregados()
print(f"Acidentes unicos carregados: {len(df_acidentes_base):,}")
print(f"Memoria da base agregada: {df_acidentes_base.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Acidentes unicos carregados: 604,223
Memoria da base agregada: 63.46 MB


## 7. Separacao dos dataframes

As chaves `sk_*` sao chaves tecnicas sequenciais para facilitar joins analiticos. O campo original `id` permanece nas tabelas 1:N para rastreabilidade e ligacao com a fato.

- `df_fato_acidentes`: uma linha por acidente, com chaves e metricas principais.
- `df_dim_tempo` e `df_dim_localidade`: dimensoes deduplicadas ligadas por chave tecnica.
- `df_dim_veiculo`, `df_dim_envolvido`, `df_dim_circunstancia` e `df_dim_causas`: tabelas auxiliares que preservam os detalhes potencialmente multiplos por acidente.

In [7]:
COLS_TEMPO = ["data_inversa", "dia_semana", "horario", "data_hora", "fase_dia"]
COLS_LOCALIDADE = ["uf", "br", "km", "municipio", "latitude", "longitude", "regional", "delegacia", "uop"]

df_dim_tempo = df_acidentes_base[COLS_TEMPO].drop_duplicates().reset_index(drop=True)
df_dim_tempo.insert(0, "sk_tempo", pd.RangeIndex(1, len(df_dim_tempo) + 1))
df_dim_tempo["ano"] = df_dim_tempo["data_inversa"].dt.year.astype("Int16")
df_dim_tempo["mes"] = df_dim_tempo["data_inversa"].dt.month.astype("Int8")
df_dim_tempo["dia"] = df_dim_tempo["data_inversa"].dt.day.astype("Int8")

df_dim_localidade = df_acidentes_base[COLS_LOCALIDADE].drop_duplicates().reset_index(drop=True)
df_dim_localidade.insert(0, "sk_localidade", pd.RangeIndex(1, len(df_dim_localidade) + 1))

df_fato_acidentes = (
    df_acidentes_base
    .merge(df_dim_tempo, on=COLS_TEMPO, how="left", validate="many_to_one")
    .merge(df_dim_localidade, on=COLS_LOCALIDADE, how="left", validate="many_to_one")
)
df_fato_acidentes.insert(0, "sk_acidente", pd.RangeIndex(1, len(df_fato_acidentes) + 1))
df_fato_acidentes = df_fato_acidentes[
    ["sk_acidente", "id", "sk_tempo", "sk_localidade", "classificacao_acidente",
     "ilesos", "feridos_leves", "feridos_graves", "mortos"]
]

df_fato_acidentes.head()

,sk_acidente,id,sk_tempo,sk_localidade,classificacao_acidente,ilesos,feridos_leves,feridos_graves,mortos
0,1,5,1,1,<NA>,<NA>,<NA>,<NA>,<NA>
1,2,8,2,2,Com Vítimas Feridas,0,0,1,0
2,3,9,3,3,Sem Vítimas,1,0,0,0
3,4,11,2,4,Com Vítimas Feridas,0,1,0,0
4,5,12,2,5,Com Vítimas Feridas,0,0,1,0


In [8]:
df_dim_veiculo = carregar_unicos(
    ["id", "id_veiculo", "tipo_veiculo", "marca", "ano_fabricacao_veiculo"],
    subset=["id_veiculo"],
    filtrar="id_veiculo",
)
df_dim_veiculo.insert(0, "sk_veiculo", pd.RangeIndex(1, len(df_dim_veiculo) + 1))

df_dim_envolvido = carregar_unicos(
    ["id", "pesid", "id_veiculo", "tipo_envolvido", "estado_fisico", "idade", "sexo"],
    subset=["pesid"],
    filtrar="pesid",
)
df_dim_envolvido.insert(0, "sk_envolvido", pd.RangeIndex(1, len(df_dim_envolvido) + 1))

df_dim_circunstancia = carregar_unicos(
    ["id", "ordem_tipo_acidente", "tipo_acidente", "classificacao_acidente", "sentido_via",
     "condicao_metereologica", "tipo_pista", "tracado_via", "uso_solo"],
    subset=["id", "ordem_tipo_acidente", "tipo_acidente", "sentido_via", "condicao_metereologica",
            "tipo_pista", "tracado_via", "uso_solo"],
    filtrar="id",
)
df_dim_circunstancia.insert(0, "sk_circunstancia", pd.RangeIndex(1, len(df_dim_circunstancia) + 1))

df_dim_causas = carregar_unicos(
    ["id", "causa_principal", "causa_acidente"],
    subset=["id", "causa_principal", "causa_acidente"],
    filtrar="id",
)
df_dim_causas.insert(0, "sk_causa", pd.RangeIndex(1, len(df_dim_causas) + 1))

qtd_veiculos = df_dim_veiculo.groupby("id", observed=True).size().rename("qtd_veiculos")
qtd_envolvidos = df_dim_envolvido.groupby("id", observed=True).size().rename("qtd_envolvidos")
df_fato_acidentes = df_fato_acidentes.merge(qtd_veiculos, on="id", how="left").merge(qtd_envolvidos, on="id", how="left")
df_fato_acidentes[["qtd_veiculos", "qtd_envolvidos"]] = (
    df_fato_acidentes[["qtd_veiculos", "qtd_envolvidos"]].fillna(0).astype("Int32")
)

df_resumo_modelagem = pd.DataFrame(
    {
        "dataframe": [
            "df_fato_acidentes", "df_dim_tempo", "df_dim_localidade", "df_dim_veiculo",
            "df_dim_envolvido", "df_dim_circunstancia", "df_dim_causas",
        ],
        "linhas": [
            len(df_fato_acidentes), len(df_dim_tempo), len(df_dim_localidade), len(df_dim_veiculo),
            len(df_dim_envolvido), len(df_dim_circunstancia), len(df_dim_causas),
        ],
        "colunas": [
            len(df_fato_acidentes.columns), len(df_dim_tempo.columns), len(df_dim_localidade.columns),
            len(df_dim_veiculo.columns), len(df_dim_envolvido.columns), len(df_dim_circunstancia.columns),
            len(df_dim_causas.columns),
        ],
    }
)
df_resumo_modelagem

,dataframe,linhas,colunas
0,df_fato_acidentes,604223,11
1,df_dim_tempo,424794,9
2,df_dim_localidade,415309,10
3,df_dim_veiculo,1136768,6
4,df_dim_envolvido,1439831,8
5,df_dim_circunstancia,917810,10
6,df_dim_causas,818994,4


## 8. Conferencia das causas de acidentes

As colunas reais usadas sao:

- `causa_acidente`: descricao da causa presumivel;
- `causa_principal`: indica se aquela causa e a principal do acidente;
- `id`: liga cada registro de causa ao acidente.

Como um acidente pode possuir mais de uma causa, a frequencia abaixo considera pares unicos de acidente e causa. Isso evita contar repeticoes causadas por varios envolvidos ou veiculos.

In [9]:
COLUNAS_CAUSA = ["id", "causa_principal", "causa_acidente"]
df_dicionario_causas = df_dicionario[df_dicionario["coluna"].isin(COLUNAS_CAUSA)]

causas_validas = df_dim_causas["causa_acidente"].dropna().astype("string").str.strip()
df_inconsistencias_causas = pd.DataFrame(
    {
        "verificacao": [
            "causa_acidente nula",
            "causa_acidente vazia",
            "causa_principal fora de Sim/Nao",
        ],
        "quantidade": [
            int(df_dim_causas["causa_acidente"].isna().sum()),
            int(causas_validas.eq("").sum()),
            int((~df_dim_causas["causa_principal"].astype("string").str.casefold().isin(["sim", "não", "nao"]) &
                 df_dim_causas["causa_principal"].notna()).sum()),
        ],
    }
)

df_resumo_causas = (
    df_dim_causas.dropna(subset=["causa_acidente"])
    .groupby("causa_acidente", observed=True)["id"]
    .nunique()
    .rename("quantidade")
    .sort_values(ascending=False)
    .reset_index()
)
df_resumo_causas["percentual"] = (df_resumo_causas["quantidade"] / df_resumo_causas["quantidade"].sum() * 100).round(3)

def filtrar_acidentes_por_causa(causa: str, correspondencia_exata: bool = False) -> pd.DataFrame:
    serie = df_dim_causas["causa_acidente"].astype("string")
    mascara = serie.str.casefold().eq(causa.casefold()) if correspondencia_exata else serie.str.contains(causa, case=False, na=False)
    ids = df_dim_causas.loc[mascara, "id"].drop_duplicates()
    return df_fato_acidentes[df_fato_acidentes["id"].isin(ids)].copy()

display(df_dicionario_causas)
display(df_inconsistencias_causas)
display(df_resumo_causas.head(20))
print(f"Valores unicos de causa: {df_dim_causas['causa_acidente'].nunique(dropna=True):,}")

,coluna,tipo_esperado,definicao
0,id,inteiro,Identificador do acidente.
9,causa_principal,categoria,Indica se a causa registrada e a causa principal.
10,causa_acidente,categoria,Causa presumivel do acidente.


,verificacao,quantidade
0,causa_acidente nula,9
1,causa_acidente vazia,0
2,causa_principal fora de Sim/Nao,57


,causa_acidente,quantidade,percentual
0,Falta de Atenção à Condução,122291,14.932
1,Velocidade Incompatível,82012,10.014
2,Reação tardia ou ineficiente do condutor,63374,7.738
3,Ausência de reação do condutor,55888,6.824
4,Desobediência às normas de trânsito pelo condutor,35857,4.378
5,Acessar a via sem observar a presença dos outr...,32771,4.001
6,Condutor deixou de manter distância do veículo...,30141,3.680
7,Ingestão de álcool pelo condutor,28588,3.491
8,Condutor Dormindo,26993,3.296
9,Ingestão de Álcool,25602,3.126


Valores unicos de causa: 142


In [10]:
# Exemplo reutilizavel de filtro; altere somente o texto da causa.
exemplo_causa = str(df_resumo_causas.iloc[0]["causa_acidente"])
df_acidentes_causa_exemplo = filtrar_acidentes_por_causa(exemplo_causa, correspondencia_exata=True)
print(f"Causa do exemplo: {exemplo_causa}")
print(f"Acidentes encontrados: {len(df_acidentes_causa_exemplo):,}")
df_acidentes_causa_exemplo.head()

Causa do exemplo: Falta de Atenção à Condução
Acidentes encontrados: 122,291


,sk_acidente,id,sk_tempo,sk_localidade,classificacao_acidente,ilesos,feridos_leves,feridos_graves,mortos,qtd_veiculos,qtd_envolvidos
2,3,9,3,3,Sem Vítimas,1,0,0,0,1,1
6,7,14,4,7,Com Vítimas Feridas,1,1,0,0,2,2
10,11,18,7,11,Com Vítimas Feridas,0,1,0,0,1,2
16,17,24,11,17,Com Vítimas Feridas,1,1,0,0,2,2
24,25,35,16,25,Sem Vítimas,1,0,0,0,2,2


## 9. Conferencias rapidas

Estas celulas verificam se a modelagem permite responder consultas basicas sem voltar ao dataframe bruto gigante. Nao sao analises finais; sao apenas testes de consistencia e utilidade.

In [11]:
df_acidentes_tempo = df_fato_acidentes.merge(
    df_dim_tempo[["sk_tempo", "ano", "fase_dia"]], on="sk_tempo", how="left", validate="many_to_one"
)
df_acidentes_localidade = df_fato_acidentes.merge(
    df_dim_localidade[["sk_localidade", "uf", "municipio"]], on="sk_localidade", how="left", validate="many_to_one"
)

conferencias_rapidas = {
    "acidentes_por_ano": df_acidentes_tempo.groupby("ano", observed=True)["id"].nunique().sort_index(),
    "acidentes_por_uf": df_acidentes_localidade.groupby("uf", observed=True)["id"].nunique().sort_values(ascending=False),
    "acidentes_por_municipio": df_acidentes_localidade.groupby(["uf", "municipio"], observed=True)["id"].nunique().sort_values(ascending=False),
    "acidentes_por_tipo": df_dim_circunstancia.groupby("tipo_acidente", observed=True)["id"].nunique().sort_values(ascending=False),
    "acidentes_por_causa": df_resumo_causas.set_index("causa_acidente")["quantidade"],
    "acidentes_por_clima": df_dim_circunstancia.groupby("condicao_metereologica", observed=True)["id"].nunique().sort_values(ascending=False),
    "acidentes_por_fase_dia": df_acidentes_tempo.groupby("fase_dia", observed=True)["id"].nunique().sort_values(ascending=False),
}

display(conferencias_rapidas["acidentes_por_ano"])
display(conferencias_rapidas["acidentes_por_uf"].head(10))
display(conferencias_rapidas["acidentes_por_tipo"].head(10))
display(conferencias_rapidas["acidentes_por_clima"].head(10))

df_totais_relacionados = pd.DataFrame(
    {
        "metrica": ["acidentes", "veiculos", "envolvidos"],
        "quantidade": [df_fato_acidentes["id"].nunique(), df_dim_veiculo["id_veiculo"].nunique(), df_dim_envolvido["pesid"].nunique()],
    }
)
df_totais_relacionados

ano
2017    89567
2018    69333
2019    67558
2020    63586
2021    64567
2022    64607
2023    67767
2024    73156
2025    44081
Name: id, dtype: int64

uf
MG    79620
SC    71479
PR    67613
RJ    44487
RS    42309
SP    40118
BA    32423
GO    29075
PE    24785
ES    22071
Name: id, dtype: int64

tipo_acidente
Saída de leito carroçável       130064
Tombamento                      122052
Colisão traseira                118770
Queda de ocupante de veículo    108422
Colisão transversal              77864
Capotamento                      60586
Colisão com objeto               46350
Colisão frontal                  43435
Colisão com objeto estático      38540
Colisão lateral                  37718
Name: id, dtype: int64

condicao_metereologica
Céu Claro           359722
Nublado             100288
Chuva                67711
Sol                  40647
Garoa/Chuvisco       20952
Ignorado              8531
Nevoeiro/Neblina      5163
Vento                 1178
Granizo                 20
Neve                    10
Name: id, dtype: int64

,metrica,quantidade
0,acidentes,604223
1,veiculos,1136768
2,envolvidos,1439831


## 10. Validacao final da modelagem

As assercoes abaixo tornam explicitos os contratos principais: uma linha por acidente na fato, chaves tecnicas unicas, relacionamentos resolvidos e colunas de causa coerentes com o dicionario.

In [12]:
assert df_fato_acidentes["id"].is_unique, "A fato deve possuir uma linha por acidente."
assert df_fato_acidentes["sk_acidente"].is_unique, "sk_acidente deve ser unica."
assert df_dim_tempo["sk_tempo"].is_unique, "sk_tempo deve ser unica."
assert df_dim_localidade["sk_localidade"].is_unique, "sk_localidade deve ser unica."
assert df_fato_acidentes["sk_tempo"].notna().all(), "Toda fato deve apontar para tempo."
assert df_fato_acidentes["sk_localidade"].notna().all(), "Toda fato deve apontar para localidade."
assert set(COLUNAS_CAUSA).issubset(df_dim_causas.columns), "Colunas reais de causa ausentes."
assert set(COLUNAS_CAUSA).issubset(set(df_dicionario["coluna"])), "Colunas de causa ausentes no dicionario."
assert not df_validacao_colunas["colunas_ausentes"].any(), "Existem colunas obrigatorias ausentes."

print("VALIDACAO FINAL: todos os contratos principais foram atendidos.")
df_resumo_modelagem

VALIDACAO FINAL: todos os contratos principais foram atendidos.


,dataframe,linhas,colunas
0,df_fato_acidentes,604223,11
1,df_dim_tempo,424794,9
2,df_dim_localidade,415309,10
3,df_dim_veiculo,1136768,6
4,df_dim_envolvido,1439831,8
5,df_dim_circunstancia,917810,10
6,df_dim_causas,818994,4


## 11. Salvamento opcional dos dataframes

O salvamento fica desativado por padrao para nao gerar artefatos grandes durante toda execucao. Ao definir `SALVAR_PARQUETS = True`, os dataframes sao gravados em `data/processed/modelagem_dimensional/`, sem sobrescrever os CSVs ou Parquets originais.

In [13]:
SALVAR_PARQUETS = False
OUTPUT_DIR = PROCESSED_DIR / "modelagem_dimensional"

dataframes_modelagem = {
    "fato_acidentes": df_fato_acidentes,
    "dim_tempo": df_dim_tempo,
    "dim_localidade": df_dim_localidade,
    "dim_veiculo": df_dim_veiculo,
    "dim_envolvido": df_dim_envolvido,
    "dim_circunstancia": df_dim_circunstancia,
    "dim_causas": df_dim_causas,
}

if SALVAR_PARQUETS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for nome, dataframe in dataframes_modelagem.items():
        caminho = OUTPUT_DIR / f"{nome}.parquet"
        dataframe.to_parquet(caminho, index=False)
        print(f"Salvo: {caminho.relative_to(ROOT)}")
else:
    print("Salvamento opcional desativado; nenhum arquivo de dados foi criado.")

Salvamento opcional desativado; nenhum arquivo de dados foi criado.


## Conclusao

A base esta validada e organizada para futuras analises. A tabela fato mantem uma linha por acidente e as tabelas auxiliares permitem consultar tempo, localidade, veiculos, envolvidos, circunstancias e causas sem misturar todos os detalhes em um unico dataframe gigante.